# CodeDoctor: AppsRetrieval evaluation (submission configuration)

Runs `scripts/evaluate_mteb.py`, the official MTEB `AppsRetrieval` evaluation, with
`config/submission.yaml`: F2LLM-v2-1.7B, **queries encoded on the CPU**, and the
accepted execution-verification settings. The pipeline source is embedded below,
generated from the repository by `scripts/make_notebook.py`.

Document vectors are built once, offline (the FAQ allows a GPU for pre-processing),
and the evaluation refuses to run if any document is missing from that cache.
Encoding 3,765 queries with a 1.7B model on a CPU takes many hours; `--query-store`
checkpoints each vector so an interrupted run resumes instead of restarting.

The evaluation **executes Python that came from the dataset** in a sandboxed
subprocess (a resource limiter, not a security boundary: no network or filesystem
isolation). Run it in a disposable environment.


In [ ]:
# 1. Environment
import os, platform, psutil, torch
print(platform.platform())
print('cores', psutil.cpu_count(logical=False), '/', os.cpu_count(),
      ' RAM', round(psutil.virtual_memory().total / 1e9, 1), 'GB')
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())


In [ ]:
# 2. Unpack the pipeline (generated from the repository)
PAYLOAD = "..."

import base64, io, sys, tarfile

WORK = os.environ.get('CODEDOCTOR_WORK', '/content/codedoctor')
os.makedirs(WORK, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(PAYLOAD)), mode='r:gz') as tar:
    tar.extractall(WORK)
sys.path.insert(0, WORK)
os.chdir(WORK)
print(sorted(os.listdir(WORK)))


In [ ]:
# 3. Pinned requirements (mteb decides the result schema and the dataset revision)
!pip install -q -r requirements-submission.txt


## Offline document cache

Encodes the 8,765 corpus documents once and records the model, revision, dtype and
checksums in `documents/manifest.json`. Use a GPU if there is one (about ten minutes on a
T4); on a CPU this step alone takes hours.


In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('document cache on', DEVICE)
!python scripts/prepare_document_cache.py --device $DEVICE --out documents


## The evaluation

Query inference is on the CPU. Results go to `results/` (an existing output is never
overwritten without `--overwrite`).


In [ ]:
!python scripts/evaluate_mteb.py --device cpu \
    --corpus-cache documents/corpus.sqlite --require-complete-corpus-cache \
    --query-store results/queries.sqlite \
    --predictions results/predictions --output results/appsretrieval_results.json
!python scripts/validate_submission.py results/appsretrieval_results.json


In [ ]:
# Download the artifacts (Colab); on Kaggle they are already under /kaggle/working
try:
    from google.colab import files  # noqa: F401
    for name in ('results/appsretrieval_results.json', 'results/submission_predictions.csv',
                 'results/submission_metrics.csv', 'results/execution_audit.json',
                 'documents/manifest.json'):
        if os.path.exists(name):
            files.download(name)
except ImportError:
    print('not on Colab: collect the files under', os.path.abspath('results'))
